# 📊 Ploidia Metrics Comparison: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality and validation queries on the ploidia tables:
- Local table: `huntington_data_lake.gold.data_ploidia`
- Cloud table: `gold_huntington_prod.data_ploidia_metrics_enriched`

Per project requirements, image availability columns (`api_response_code` and `api_error_message`) are skipped.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
import re
import math
warnings.filterwarnings('ignore')

# 1. Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
try:
    ath_con = connect(
        region_name="sa-east-1",
        work_group="datalake-admins"
    )
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_con = None

In [ ]:
def normalize_column_name(col):
    """Normalize column names to lowercase with underscores"""
    if not isinstance(col, str):
        return col
    c = col.lower().strip()
    c = re.sub(r'[-\s\.]+', '_', c)
    c = c.strip('_')
    return c

def normalize_value(value, column_name=None):
    """
    Normalize values to ignore language differences, rounding, and type differences.
    Adapted from 04_compare_excel_with_database.py
    """
    if pd.isna(value) or value is None:
        return None
    
    # Special handling for Patient ID: convert to integer
    if column_name in ["patient_id", "patient id"]:
        try:
            if isinstance(value, str):
                clean_value = value.replace('.', '').replace(',', '').strip()
                return int(clean_value)
            elif isinstance(value, float):
                if value < 1000:
                    return int(value * 1000)
                return int(value)
            return int(value)
        except (ValueError, TypeError):
            return value
            
    # Special handling for Video ID/Slide ID/Embryo ID: strip well suffix
    if column_name in ["video_id", "slide_id", "embryo_id", "video id", "slide id", "embryo id"]:
        if isinstance(value, str):
            return re.sub(r'-\d+$', '', value).strip()
        return value
        
    # Try converting to float and truncate to 1 decimal place
    try:
        numeric_value = float(value)
        return math.floor(numeric_value * 10) / 10
    except (ValueError, TypeError):
        pass
        
    # Normalize string: lowercase, strip, spacing and translations
    if isinstance(value, str):
        normalized = value.lower().strip()
        normalized = re.sub(r'\s*,\s*', ', ', normalized)
        normalized = re.sub(r'\s*/\s*', '/', normalized)
        
        language_map = {
            'fresco': 'fresh',
            'fresh': 'fresh',
            'descongelado': 'thawed',
            'thawed': 'thawed',
            'autólogo': 'autologous',
            'autologous': 'autologous',
            'homólogo': 'autologous',
            'homologous': 'autologous',
            'heterólogo': 'donor',
            'heterologous': 'donor',
            'donor': 'donor',
            'ambos': 'both',
            'both': 'both',
            'ibirapuera': 'ibi',
            'ibi': 'ibi',
            'fator masculino': 'male factor',
            'male factor': 'male factor',
            'fator feminino': 'female factor',
            'female factor': 'female factor',
            'fator misto': 'mixed factor',
            'mixed factor': 'mixed factor',
            'idiopático': 'idiopathic',
            'idiopathic': 'idiopathic',
            'outros': 'other',
            'other': 'other',
            'endometriose': 'endometriosis',
            'endometriosis': 'endometriosis',
            'inexplicado': 'unknown',
            'unknown': 'unknown',
        }
        return language_map.get(normalized, normalized)
        
    return value

def extract_year_from_slide_id(slide_id):
    """Extract year YYYY from Slide ID formatted like DYYYY.MM.DD_..."""
    if not isinstance(slide_id, str):
        return None
    match = re.search(r'D(\d{4})\.', slide_id)
    if match:
        return int(match.group(1))
    return None

## 🔍 Part 1: DuckDB Queries (Local Ploidia Table)

Running local metrics on `gold.data_ploidia` in DuckDB.

In [ ]:
# Fetch DuckDB data
df_duck = duck_con.execute("SELECT * FROM gold.data_ploidia").df()
df_duck.columns = [normalize_column_name(col) for col in df_duck.columns]

# Exclude image availability columns
exclude_cols = ['api_response_code', 'api_error_message']
df_duck = df_duck.drop(columns=[col for col in exclude_cols if col in df_duck.columns])

# Extract year and clean location
df_duck["year"] = df_duck["slide_id"].apply(extract_year_from_slide_id)
df_duck["unidade"] = df_duck["unidade"].str.strip() if "unidade" in df_duck.columns else "Unknown"

print(f"Loaded {len(df_duck)} rows from local DuckDB gold.data_ploidia.")

## 🔍 Part 2: AWS Athena Queries (Cloud Ploidia Table)

Running cloud metrics on `gold_huntington_prod.data_ploidia_metrics_enriched` in AWS Athena.

In [ ]:
if ath_con is not None:
    df_ath = pd.read_sql("SELECT * FROM gold_huntington_prod.data_ploidia_metrics_enriched", ath_con)
    df_ath.columns = [normalize_column_name(col) for col in df_ath.columns]
    
    # Exclude image availability columns
    df_ath = df_ath.drop(columns=[col for col in exclude_cols if col in df_ath.columns])
    
    # Extract year and clean location
    df_ath["year"] = df_ath["slide_id"].apply(extract_year_from_slide_id)
    df_ath["unidade"] = df_ath["unidade"].str.strip() if "unidade" in df_ath.columns else "Unknown"
    
    print(f"Loaded {len(df_ath)} rows from cloud Athena gold_huntington_prod.data_ploidia_metrics_enriched.")
else:
    df_ath = None
    print("AWS Athena connection is not available.")

## ⚖️ Part 3: Side-by-Side Reconciliation Tables

Merging and displaying the local (DuckDB) and cloud (Athena) metrics side-by-side to highlight differences.

In [ ]:
if df_ath is not None:
    print("--- Side-by-side Overall Comparison ---")
    duck_overall = {
        "source": "Local (DuckDB)",
        "total_rows": len(df_duck),
        "unique_patients": df_duck["patient_id"].nunique(),
        "unique_embryos": df_duck["embryo_id"].nunique(),
        "unique_slides": df_duck["slide_id"].nunique()
    }
    
    ath_overall = {
        "source": "Cloud (Athena)",
        "total_rows": len(df_ath),
        "unique_patients": df_ath["patient_id"].nunique(),
        "unique_embryos": df_ath["embryo_id"].nunique(),
        "unique_slides": df_ath["slide_id"].nunique()
    }
    
    df_overall = pd.DataFrame([duck_overall, ath_overall])
    display(df_overall)
else:
    print("Athena data not available.")

In [ ]:
if df_ath is not None:
    print("--- Side-by-side Yearly Match Comparison ---")
    duck_yr = df_duck.groupby("year").agg(
        duck_rows=("patient_id", "count"),
        duck_patients=("patient_id", "nunique"),
        duck_embryos=("embryo_id", "nunique")
    ).reset_index()
    
    ath_yr = df_ath.groupby("year").agg(
        ath_rows=("patient_id", "count"),
        ath_patients=("patient_id", "nunique"),
        ath_embryos=("embryo_id", "nunique")
    ).reset_index()
    
    df_yr_comp = pd.merge(duck_yr, ath_yr, on="year", how="outer").sort_values("year").fillna(0)
    for col in df_yr_comp.columns:
        if col != "year":
            df_yr_comp[col] = df_yr_comp[col].astype(int)
    display(df_yr_comp)
else:
    print("Athena data not available.")

In [ ]:
if df_ath is not None:
    print("--- Side-by-side Location Breakdown Comparison ---")
    duck_loc = df_duck.groupby("unidade").agg(
        duck_rows=("patient_id", "count"),
        duck_patients=("patient_id", "nunique")
    ).reset_index()
    
    ath_loc = df_ath.groupby("unidade").agg(
        ath_rows=("patient_id", "count"),
        ath_patients=("patient_id", "nunique")
    ).reset_index()
    
    df_loc_comp = pd.merge(duck_loc, ath_loc, on="unidade", how="outer").fillna(0)
    for col in df_loc_comp.columns:
        if col != "unidade":
            df_loc_comp[col] = df_loc_comp[col].astype(int)
    display(df_loc_comp)
else:
    print("Athena data not available.")

In [ ]:
if df_ath is not None:
    print("--- Side-by-side Column Data Completion Comparison ---")
    common_cols = sorted(list(set(df_duck.columns) & set(df_ath.columns)))
    
    completion_records = []
    for col in common_cols:
        if col in ["year"]:
            continue
        
        # DuckDB non-null / non-blank
        duck_valid = df_duck[col].dropna()
        duck_valid = duck_valid[duck_valid.astype(str).str.strip() != ""] if df_duck[col].dtype == object else duck_valid
        duck_pct = (len(duck_valid) / len(df_duck)) * 100.0 if len(df_duck) > 0 else 0.0
        
        # Athena non-null / non-blank
        ath_valid = df_ath[col].dropna()
        ath_valid = ath_valid[ath_valid.astype(str).str.strip() != ""] if df_ath[col].dtype == object else ath_valid
        ath_pct = (len(ath_valid) / len(df_ath)) * 100.0 if len(df_ath) > 0 else 0.0
        
        completion_records.append({
            "column": col,
            "duck_non_null": len(duck_valid),
            "duck_completion_pct": round(duck_pct, 2),
            "ath_non_null": len(ath_valid),
            "ath_completion_pct": round(ath_pct, 2),
            "diff_pct": round(abs(duck_pct - ath_pct), 2)
        })
        
    df_completion = pd.DataFrame(completion_records).sort_values("diff_pct", ascending=False)
    display(df_completion)
else:
    print("Athena data not available.")

## 🔍 Part 4: Row-Level Value Discrepancy Analysis

Inner joining the datasets on `patient_id` + `slide_id` + `well` to check for exact value discrepancies in each matching column.

In [ ]:
if df_ath is not None:
    print("--- Row-level inner join stats ---")
    merged = pd.merge(
        df_duck,
        df_ath,
        on=["patient_id", "slide_id", "well"],
        suffixes=("_duck", "_ath"),
        how="inner"
    )
    print(f"Inner joined matching rows count: {len(merged)}")
    print(f"Rows only in DuckDB: {len(df_duck) - len(merged)}")
    print(f"Rows only in Athena: {len(df_ath) - len(merged)}")
    
    value_discrepancy_cols = [c for c in common_cols if c not in ["patient_id", "slide_id", "well", "year"]]
    discrepancy_counts = {}
    
    for col in value_discrepancy_cols:
        col_duck = f"{col}_duck"
        col_ath = f"{col}_ath"
        
        # Apply normalization to both columns
        norm_duck = merged[col_duck].apply(lambda x: normalize_value(x, col))
        norm_ath = merged[col_ath].apply(lambda x: normalize_value(x, col))
        
        # Find rows where they do not match
        mask = (norm_duck != norm_ath) & ~(norm_duck.isna() & norm_ath.isna())
        diff_count = mask.sum()
        
        if diff_count > 0:
            discrepancy_counts[col] = diff_count
            
    if discrepancy_counts:
        print("\nFound value discrepancies in the following columns:")
        for col, count in sorted(discrepancy_counts.items(), key=lambda x: x[1], reverse=True):
            print(f" - {col}: {count} rows with differences ({round(count * 100.0 / len(merged), 2)}% of matched)")
            
            # Print a few samples
            col_duck = f"{col}_duck"
            col_ath = f"{col}_ath"
            norm_duck = merged[col_duck].apply(lambda x: normalize_value(x, col))
            norm_ath = merged[col_ath].apply(lambda x: normalize_value(x, col))
            mask = (norm_duck != norm_ath) & ~(norm_duck.isna() & norm_ath.isna())
            samples = merged[mask][["patient_id", "slide_id", "well", col_duck, col_ath]].head(5)
            print("   Sample discrepancies:")
            display(samples)
            print()
    else:
        print("\nSuccess! No value discrepancies found among the inner joined rows.")
else:
    print("Athena data not available.")